In [1]:
import pandas as pd
import numpy as np

from pathlib import Path

from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

In [6]:
processed_path = Path("../data/processed")

customers = pd.read_parquet(processed_path / "customers_clean.parquet")
orders = pd.read_parquet(processed_path / "orders_clean.parquet")
transactions = pd.read_parquet(processed_path / "transactions_clean.parquet")

print("Customers:", customers.shape)
print("Orders:", orders.shape)
print("Transactions:", transactions.shape)

Customers: (100, 5)
Orders: (200, 4)
Transactions: (200, 6)


In [7]:
print("Customers columns:")
print(customers.columns)

print("\nOrders columns:")
print(orders.columns)

print("\nTransactions columns:")
print(transactions.columns)

Customers columns:
Index(['customer_id', 'first_name', 'email', 'city', 'signup_date'], dtype='str')

Orders columns:
Index(['order_id', 'customer_id', 'order_date', 'status'], dtype='str')

Transactions columns:
Index(['transaction_id', 'order_id', 'payment_method', 'payment_status',
       'transaction_date', 'amount'],
      dtype='str')


In [8]:
customer_orders = orders.groupby("customer_id").agg(
    total_orders=("order_id", "count")
).reset_index()

customer_orders.head()

,customer_id,total_orders
0,2,2
1,3,2
2,8,3
3,9,3
4,10,3


In [9]:
customer_orders["will_reorder"] = (customer_orders["total_orders"] > 1).astype(int)

customer_orders.head()

,customer_id,total_orders,will_reorder
0,2,2,1
1,3,2,1
2,8,3,1
3,9,3,1
4,10,3,1


In [10]:
order_transactions = transactions.merge(
    orders[["order_id", "customer_id"]],
    on="order_id",
    how="left"
)

order_transactions.head()

,transaction_id,order_id,payment_method,payment_status,transaction_date,amount,customer_id
0,1,1,card,paid,2026-01-04 17:14:47.555098,2467.21,25
1,2,2,cash,failed,2026-02-08 17:14:47.555110,3263.99,27
2,3,3,card,refunded,2026-03-25 17:14:47.555115,4431.36,63
3,4,4,bank_transfer,refunded,2026-02-20 17:14:47.555118,3453.62,93
4,5,5,cash,refunded,2025-11-28 17:14:47.555122,3625.70,81


In [11]:
customer_spending = order_transactions.groupby("customer_id").agg(
    total_spent=("amount", "sum"),
    average_order_amount=("amount", "mean"),
    successful_payments=("payment_status", lambda x: (x == "paid").sum())
).reset_index()

customer_spending.head()

,customer_id,total_spent,average_order_amount,successful_payments
0,2,5316.66,2658.330000,0
1,3,5668.17,2834.085000,1
2,8,13999.28,4666.426667,1
3,9,13896.10,4632.033333,1
4,10,10051.18,3350.393333,1


In [12]:
ml_data = customers.merge(customer_orders, on="customer_id", how="left")
ml_data = ml_data.merge(customer_spending, on="customer_id", how="left")

ml_data.head()

,customer_id,first_name,email,city,signup_date,total_orders,will_reorder,total_spent,average_order_amount,successful_payments
0,1,Customer1,customer1@example.com,Abu Dhabi,2025-09-22 17:14:47.542946,NaN,NaN,NaN,NaN,NaN
1,2,Customer2,customer2@example.com,Ajman,2025-11-21 17:14:47.542968,2.0,1.0,5316.66,2658.330,0.0
2,3,Customer3,customer3@example.com,Abu Dhabi,2026-02-21 17:14:47.542973,2.0,1.0,5668.17,2834.085,1.0
3,4,Customer4,customer4@example.com,Sharjah,2025-10-09 17:14:47.542977,NaN,NaN,NaN,NaN,NaN
4,5,Customer5,customer5@example.com,Dubai,2025-07-22 17:14:47.542982,NaN,NaN,NaN,NaN,NaN


In [13]:
ml_data.isnull().sum()

customer_id              0
first_name               0
email                    0
city                     0
signup_date              0
total_orders            14
will_reorder            14
total_spent             14
average_order_amount    14
successful_payments     14
dtype: int64

In [14]:
ml_data["total_orders"] = ml_data["total_orders"].fillna(0)
ml_data["will_reorder"] = ml_data["will_reorder"].fillna(0)
ml_data["total_spent"] = ml_data["total_spent"].fillna(0)
ml_data["average_order_amount"] = ml_data["average_order_amount"].fillna(0)
ml_data["successful_payments"] = ml_data["successful_payments"].fillna(0)

ml_data.isnull().sum()

customer_id             0
first_name              0
email                   0
city                    0
signup_date             0
total_orders            0
will_reorder            0
total_spent             0
average_order_amount    0
successful_payments     0
dtype: int64

In [15]:
features = [
    "city",
    "total_orders",
    "total_spent",
    "average_order_amount",
    "successful_payments"
]

target = "will_reorder"

X = ml_data[features]
y = ml_data[target]

X.head()

,city,total_orders,total_spent,average_order_amount,successful_payments
0,Abu Dhabi,0.0,0.00,0.000,0.0
1,Ajman,2.0,5316.66,2658.330,0.0
2,Abu Dhabi,2.0,5668.17,2834.085,1.0
3,Sharjah,0.0,0.00,0.000,0.0
4,Dubai,0.0,0.00,0.000,0.0


In [16]:
numeric_features = [
    "total_orders",
    "total_spent",
    "average_order_amount",
    "successful_payments"
]

categorical_features = [
    "city"
]

print("Numeric features:", numeric_features)
print("Categorical features:", categorical_features)

Numeric features: ['total_orders', 'total_spent', 'average_order_amount', 'successful_payments']
Categorical features: ['city']


In [17]:
numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

In [18]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipeline, numeric_features),
        ("cat", categorical_pipeline, categorical_features)
    ]
)

In [19]:
X_processed = preprocessor.fit_transform(X)

X_processed.shape

(100, 8)

In [20]:
feature_names = preprocessor.get_feature_names_out()

feature_names

array(['num__total_orders', 'num__total_spent',
       'num__average_order_amount', 'num__successful_payments',
       'cat__city_Abu Dhabi', 'cat__city_Ajman', 'cat__city_Dubai',
       'cat__city_Sharjah'], dtype=object)

In [21]:
features_df = pd.DataFrame(
    X_processed.toarray() if hasattr(X_processed, "toarray") else X_processed,
    columns=feature_names
)

features_df.head()

,num__total_orders,num__total_spent,num__average_order_amount,num__successful_payments,cat__city_Abu Dhabi,cat__city_Ajman,cat__city_Dubai,cat__city_Sharjah
0,-1.360828,-1.217951,-1.534777,-0.928142,1.0,0.0,0.0,0.0
1,0.000000,-0.356606,-0.296974,-0.928142,0.0,1.0,0.0,0.0
2,0.000000,-0.299658,-0.215137,0.457144,1.0,0.0,0.0,0.0
3,-1.360828,-1.217951,-1.534777,-0.928142,0.0,0.0,0.0,1.0
4,-1.360828,-1.217951,-1.534777,-0.928142,0.0,0.0,1.0,0.0


In [22]:
features_df["will_reorder"] = y.values

features_df.head()

,num__total_orders,num__total_spent,num__average_order_amount,num__successful_payments,cat__city_Abu Dhabi,cat__city_Ajman,cat__city_Dubai,cat__city_Sharjah,will_reorder
0,-1.360828,-1.217951,-1.534777,-0.928142,1.0,0.0,0.0,0.0,0.0
1,0.000000,-0.356606,-0.296974,-0.928142,0.0,1.0,0.0,0.0,1.0
2,0.000000,-0.299658,-0.215137,0.457144,1.0,0.0,0.0,0.0,1.0
3,-1.360828,-1.217951,-1.534777,-0.928142,0.0,0.0,0.0,1.0,0.0
4,-1.360828,-1.217951,-1.534777,-0.928142,0.0,0.0,1.0,0.0,0.0


In [23]:
output_path = Path("../data/processed/features.csv")

features_df.to_csv(output_path, index=False)

print(f"Saved processed features to: {output_path}")

Saved processed features to: ..\data\processed\features.csv


In [24]:
check_df = pd.read_csv("../data/processed/features.csv")

print(check_df.shape)
check_df.head()

(100, 9)


,num__total_orders,num__total_spent,num__average_order_amount,num__successful_payments,cat__city_Abu Dhabi,cat__city_Ajman,cat__city_Dubai,cat__city_Sharjah,will_reorder
0,-1.360828,-1.217951,-1.534777,-0.928142,1.0,0.0,0.0,0.0,0.0
1,0.000000,-0.356606,-0.296974,-0.928142,0.0,1.0,0.0,0.0,1.0
2,0.000000,-0.299658,-0.215137,0.457144,1.0,0.0,0.0,0.0,1.0
3,-1.360828,-1.217951,-1.534777,-0.928142,0.0,0.0,0.0,1.0,0.0
4,-1.360828,-1.217951,-1.534777,-0.928142,0.0,0.0,1.0,0.0,0.0


## Preprocessing Summary

In this task, I prepared the customer dataset for machine learning by creating customer-level features and applying preprocessing steps.

First, I loaded the cleaned customers, orders, and transactions tables. Then I created order and spending features such as total_orders, total_spent, average_order_amount, and successful_payments. I also created the target column will_reorder.

After creating the ML dataset, I handled missing values by replacing missing order and spending values with 0, because those customers had no related orders or transactions.

I used a preprocessing pipeline to scale numerical columns with StandardScaler and encode the categorical city column using OneHotEncoder. The final processed dataset was saved as data/processed/features.csv for later model building.

# Week 3 Day 2 — Feature Engineering & Data Preprocessing

## Goal
The goal of this task is to prepare the machine learning dataset by handling missing values, encoding categorical columns, scaling numerical columns, and saving the final processed features for later model building.